# Small-Scale Solar Magnetic Fields — Implementation
# 소규모 태양 자기장 — 구현

**Paper.** A.G. de Wijn, J.O. Stenflo, S.K. Solanki, S. Tsuneta, *Space Sci. Rev.* **144**, 275 (2009). DOI: 10.1007/s11214-008-9473-6.

**English.** This notebook reproduces three quantitative pillars of the review:
1. Magnetic flux distribution at the quiet Sun (lognormal vs. exponential PDF for network/internetwork features).
2. Internetwork carpet flux *replacement (replenishment) timescale* given Hinode-era emergence rates.
3. A Stokes V signal model in the weak-field limit, illustrating cancellation by mixed polarities (the Zeeman vs. Hanle puzzle).

**한국어.** 본 노트북은 본 리뷰의 세 가지 정량적 기둥을 재현합니다:
1. 조용한 태양의 자속 분포(네트워크/네트워크간 feature에 대한 로그정규 vs. 지수 PDF).
2. Hinode 시대 emergence 비율을 주어진 네트워크간 카펫 *교체(replenishment) 시간 척도*.
3. 약자기장 한계의 Stokes V 신호 모델 — 혼합 극성에 의한 상쇄 시각화 (Zeeman vs. Hanle 퍼즐).

**Setup / 환경.** `study-with-ai` conda environment, `python3` kernel, NumPy + Matplotlib + SciPy.

In [ ]:
"""Imports and global plotting style.

Sets up scientific stack and a clean Matplotlib style for publication-style figures.
"""

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(seed=42)
plt.rcParams.update({
    "figure.figsize": (8, 5),
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.labelsize": 11,
    "axes.titlesize": 12,
})
print("NumPy:", np.__version__)
print("Random seed: 42 (reproducibility)")

## Part 1 — Magnetic Flux Distribution at the Quiet Sun
## Part 1 — 조용한 태양의 자속 분포

**English.** Following Hagenaar (2001) and Wang et al. (1995), we compare two empirical PDFs for individual feature unsigned flux $\Phi$ (in Mx):
- **Exponential** (Hagenaar): $p(\Phi) = \frac{1}{\Phi_0}\exp(-\Phi/\Phi_0)$ with $\Phi_0 \approx 1\times 10^{18}$ Mx for network.
- **Lognormal** (often invoked for combined network+internetwork): $p(\Phi) = \frac{1}{\Phi \sigma\sqrt{2\pi}}\exp[-(\ln\Phi - \mu)^2/(2\sigma^2)]$.

We sample $N=10^5$ features from each, plot histograms, and integrate $\int \Phi\, p(\Phi)\, d\Phi$ via `np.trapezoid` to compare mean fluxes.

**한국어.** Hagenaar(2001)과 Wang et al.(1995)을 따라, 개별 feature unsigned 자속 $\Phi$ (Mx 단위)에 대한 두 경험적 PDF를 비교합니다:
- **지수**(Hagenaar): $p(\Phi) = \frac{1}{\Phi_0}\exp(-\Phi/\Phi_0)$, 네트워크의 경우 $\Phi_0 \approx 1\times 10^{18}$ Mx.
- **로그정규**(네트워크+네트워크간 결합에 자주 사용): $p(\Phi) = \frac{1}{\Phi \sigma\sqrt{2\pi}}\exp[-(\ln\Phi - \mu)^2/(2\sigma^2)]$.

$N=10^5$ feature를 각각 샘플링, 히스토그램을 그리고, $\int \Phi\, p(\Phi)\, d\Phi$를 `np.trapezoid`로 적분하여 평균 자속을 비교합니다.

In [ ]:
"""Sample exponential and lognormal flux distributions and compare moments.

PHI0 sets the exponential e-folding scale (network).
MU and SIGMA define the lognormal in ln(Mx).
"""

N_FEATURES = 100_000
PHI0 = 1.0e18  # Hagenaar 2001 e-folding for network features (Mx)
MU = np.log(1.0e17)  # lognormal median ~ 1e17 Mx (typical internetwork scale)
SIGMA = 1.0  # spread in ln-flux

phi_exp = rng.exponential(scale=PHI0, size=N_FEATURES)
phi_log = rng.lognormal(mean=MU, sigma=SIGMA, size=N_FEATURES)

print(f"Exponential mean flux: {np.mean(phi_exp):.3e} Mx (expected {PHI0:.3e})")
print(f"Lognormal mean flux:    {np.mean(phi_log):.3e} Mx (expected {np.exp(MU + SIGMA**2/2):.3e})")
print(f"Lognormal median flux:  {np.median(phi_log):.3e} Mx (expected {np.exp(MU):.3e})")

In [ ]:
"""Plot both PDFs on log-log axes and verify normalisation via np.trapezoid.

Uses an analytic PDF curve overlaid on histogram bin centres for visual fit check.
"""

phi_grid = np.logspace(15, 20, 600)
pdf_exp_analytic = (1.0 / PHI0) * np.exp(-phi_grid / PHI0)
pdf_log_analytic = (1.0 / (phi_grid * SIGMA * np.sqrt(2 * np.pi))) * np.exp(
    -((np.log(phi_grid) - MU) ** 2) / (2 * SIGMA ** 2)
)

norm_exp = np.trapezoid(pdf_exp_analytic, phi_grid)
norm_log = np.trapezoid(pdf_log_analytic, phi_grid)
mean_exp_int = np.trapezoid(phi_grid * pdf_exp_analytic, phi_grid)
mean_log_int = np.trapezoid(phi_grid * pdf_log_analytic, phi_grid)
print(f"Exponential PDF normalisation (trapezoid): {norm_exp:.4f}")
print(f"Lognormal PDF normalisation (trapezoid):    {norm_log:.4f}")
print(f"Exponential mean (analytic integral): {mean_exp_int:.3e} Mx")
print(f"Lognormal mean (analytic integral):    {mean_log_int:.3e} Mx")

fig, ax = plt.subplots()
ax.hist(phi_exp, bins=np.logspace(15, 20, 60), density=True, alpha=0.4,
        color="tab:blue", label="Exponential samples / 지수 샘플")
ax.hist(phi_log, bins=np.logspace(15, 20, 60), density=True, alpha=0.4,
        color="tab:orange", label="Lognormal samples / 로그정규 샘플")
ax.plot(phi_grid, pdf_exp_analytic, "-", color="tab:blue", label="Exponential PDF")
ax.plot(phi_grid, pdf_log_analytic, "-", color="tab:orange", label="Lognormal PDF")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Flux per feature $\\Phi$ [Mx] / feature 당 자속")
ax.set_ylabel("Probability density $p(\\Phi)$ / 확률 밀도")
ax.set_title("Quiet-Sun feature flux distribution / 조용한 태양 feature 자속 분포")
ax.legend(loc="lower left", fontsize=9)
plt.tight_layout()
plt.show()

## Part 2 — Internetwork Magnetic Carpet Replacement Timescale
## Part 2 — 네트워크간 자기 카펫 교체 시간 척도

**English.** The replacement (replenishment) timescale is
$$\tau_{\text{rep}} = \frac{\Phi_{\text{tot}}}{\dot{\Phi}_{\text{emrg}}}$$
We vary the *resolved* mean field $\langle B \rangle$ from 5 G (pre-Hinode) to 100 G (Hanle-equivalent) and compute $\tau_{\text{rep}}$ as a function of emergence rate. This shows the carpet turnover is hours to days — *much* shorter than the 11-year cycle.

**한국어.** 교체 시간 척도는
$$\tau_{\text{rep}} = \frac{\Phi_{\text{tot}}}{\dot{\Phi}_{\text{emrg}}}$$
*분해된* 평균 자기장 $\langle B \rangle$를 5 G(Hinode 이전)부터 100 G(Hanle 환산)까지 변화시키며 emergence 비율의 함수로 $\tau_{\text{rep}}$를 계산합니다. 카펫 turnover가 시간~일 단위 — 11년 주기보다 *훨씬* 짧습니다.

In [ ]:
"""Compute replacement timescale as a function of mean field strength.

A_SUN: solar surface area in cm^2.
PHIDOT_RANGE: range of plausible emergence rates in Mx/day across the Sun.
"""

A_SUN = 4 * np.pi * (6.96e10) ** 2  # solar surface area, cm^2
B_grid = np.array([5.0, 11.0, 20.0, 60.0, 100.0])  # mean field, G
phidot_grid = np.logspace(23, 26, 200)  # emergence rate, Mx/day

fig, ax = plt.subplots()
for B_mean in B_grid:
    phi_tot = B_mean * A_SUN  # total flux assuming filling factor 1 with that mean B
    tau_days = phi_tot / phidot_grid
    ax.plot(phidot_grid, tau_days * 24.0,  # convert to hours
            label=fr"$\langle B \rangle = {B_mean:.0f}$ G")

# Hagenaar 2001 reference emergence rate
PHIDOT_HAGENAAR = 4.0e24  # Mx/day for visible hemisphere
ax.axvline(PHIDOT_HAGENAAR, color="k", ls="--", alpha=0.6,
           label="Hagenaar 2001 / 4e24 Mx/day")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Total emergence rate $\\dot{\\Phi}$ [Mx/day] / 총 emergence 비율")
ax.set_ylabel("Replacement timescale $\\tau_{\\rm rep}$ [hours] / 교체 시간 척도")
ax.set_title("Magnetic carpet replenishment / 자기 카펫 보충")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

# Tabulate at the Hagenaar rate
print("\n--- Replacement timescale at Hagenaar 2001 emergence rate ---")
for B_mean in B_grid:
    tau_h = (B_mean * A_SUN) / PHIDOT_HAGENAAR * 24.0
    print(f"<B> = {B_mean:5.0f} G --> tau_rep = {tau_h:8.2f} hours = {tau_h/24:.2f} days")

## Part 3 — Stokes V Signal Model (Weak-Field Limit)
## Part 3 — Stokes V 신호 모델 (약자기장 한계)

**English.** In the weak-field limit, the circular polarisation profile is
$$V(\lambda) \propto -\,g\, \lambda^2\, B_\parallel\, \frac{\partial I(\lambda)}{\partial \lambda}$$
We model a Gaussian absorption line centred at $\lambda_0 = 6302.5$ Å with FWHM $\sim 0.06$ Å (Fe I 630.2 nm), and demonstrate two effects:
1. **Linear scaling** of $V$ amplitude with $B_\parallel$.
2. **Cancellation** when a pixel contains equal mixed polarities — the Hanle/Zeeman puzzle.

**한국어.** 약자기장 한계에서 원편광 프로파일은
$$V(\lambda) \propto -\,g\, \lambda^2\, B_\parallel\, \frac{\partial I(\lambda)}{\partial \lambda}$$
$\lambda_0 = 6302.5$ Å에 중심을 둔 FWHM $\sim 0.06$ Å의 가우시안 흡수선(Fe I 630.2 nm)을 모델링하고 두 효과를 시연합니다:
1. $V$ 진폭의 $B_\parallel$에 대한 **선형 스케일링**.
2. 픽셀에 동일한 혼합 극성이 있을 때의 **상쇄** — Hanle/Zeeman 퍼즐.

In [ ]:
"""Construct a Gaussian Fe I 630.2 nm line and its weak-field Stokes V profile.

Returns intensity, dI/dlambda, and a callable for Stokes V given B_parallel.
"""

LAMBDA0 = 6302.5  # Å, Fe I 630.2 nm
FWHM = 0.06       # Å, typical Doppler width
SIGMA_LINE = FWHM / (2 * np.sqrt(2 * np.log(2)))
DEPTH = 0.5       # line depth (fraction of continuum)
G_LANDE = 2.5     # Landé factor for Fe I 6302.5 (g_eff)
C_ZEEMAN = 4.67e-13  # Å / (G * Å^2)

wl = np.linspace(LAMBDA0 - 0.4, LAMBDA0 + 0.4, 1000)
I_cont = 1.0
I = I_cont - DEPTH * np.exp(-((wl - LAMBDA0) ** 2) / (2 * SIGMA_LINE ** 2))
dI_dlambda = np.gradient(I, wl)

def stokes_v(b_parallel: float) -> np.ndarray:
    """Weak-field-limit Stokes V profile.

    Args:
        b_parallel: Line-of-sight magnetic field strength in Gauss.

    Returns:
        Stokes V profile sampled on the global wavelength grid `wl`.
    """
    delta_lam_b = C_ZEEMAN * G_LANDE * (LAMBDA0 ** 2) * b_parallel
    return -delta_lam_b * dI_dlambda

B_examples = [50.0, 200.0, 1000.0]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(wl, I, "k-")
axes[0].set_xlabel("Wavelength [Å] / 파장")
axes[0].set_ylabel("Stokes I / 강도")
axes[0].set_title("Fe I 6302.5 Å line / 흡수선")
for b in B_examples:
    axes[1].plot(wl, stokes_v(b), label=fr"$B_\parallel = {b:.0f}$ G")
axes[1].set_xlabel("Wavelength [Å] / 파장")
axes[1].set_ylabel("Stokes V / 원편광")
axes[1].set_title("Weak-field Stokes V / 약자기장 Stokes V")
axes[1].legend(fontsize=9)
axes[1].axhline(0, color="gray", lw=0.5)
plt.tight_layout()
plt.show()

In [ ]:
"""Demonstrate Stokes V cancellation in a mixed-polarity unresolved pixel.

We simulate a pixel composed of N_SUB sub-elements each with random sign and
magnitude drawn from |B|=B_TURB. The averaged V is compared to a |V|-based
Hanle proxy that does NOT cancel.
"""

def pixel_v_average(b_components: np.ndarray) -> np.ndarray:
    """Average Stokes V over a list of sub-pixel B_parallel components.

    Args:
        b_components: Array of line-of-sight field values for sub-pixels.

    Returns:
        Averaged Stokes V profile across the sub-pixels.
    """
    profiles = np.array([stokes_v(b) for b in b_components])
    return profiles.mean(axis=0)

N_SUB = 200
B_TURB = 100.0  # G intrinsic

# Case 1: unipolar pixel (all positive)
b_unipolar = np.full(N_SUB, B_TURB)
v_uni = pixel_v_average(b_unipolar)

# Case 2: balanced mixed polarities
signs = rng.choice([-1, 1], size=N_SUB, p=[0.5, 0.5])
b_mixed = signs * B_TURB
v_mix = pixel_v_average(b_mixed)

# Hanle proxy: averages |V|^2 (insensitive to sign) — schematic only
v_hanle_uni = np.sqrt(np.mean([stokes_v(b) ** 2 for b in b_unipolar], axis=0))
v_hanle_mix = np.sqrt(np.mean([stokes_v(b) ** 2 for b in b_mixed], axis=0))

amp_uni = np.max(np.abs(v_uni))
amp_mix = np.max(np.abs(v_mix))
amp_hanle_uni = np.max(v_hanle_uni)
amp_hanle_mix = np.max(v_hanle_mix)
print(f"Zeeman V amplitude (unipolar):       {amp_uni:.3e}")
print(f"Zeeman V amplitude (mixed polarity): {amp_mix:.3e}  -- cancellation")
print(f"Hanle proxy amplitude (unipolar):    {amp_hanle_uni:.3e}")
print(f"Hanle proxy amplitude (mixed):       {amp_hanle_mix:.3e}  -- preserved")

fig, ax = plt.subplots()
ax.plot(wl, v_uni, label="Zeeman V — unipolar / 단극성")
ax.plot(wl, v_mix, label="Zeeman V — mixed polarity / 혼합 극성")
ax.plot(wl, v_hanle_uni, ls="--", label="Hanle proxy — unipolar / 단극성")
ax.plot(wl, v_hanle_mix, ls="--", label="Hanle proxy — mixed / 혼합 극성")
ax.axhline(0, color="gray", lw=0.5)
ax.set_xlabel("Wavelength [Å] / 파장")
ax.set_ylabel("Signal / 신호")
ax.set_title("Zeeman cancellation vs. Hanle resilience / Zeeman 상쇄 vs. Hanle 강건성")
ax.legend(fontsize=8, loc="upper right")
plt.tight_layout()
plt.show()

## Part 4 — Sanity Check: Carpet Energy Budget vs. Chromospheric Heating
## Part 4 — 점검: 카펫 에너지 예산 vs. 채층 가열

**English.** Magnetic energy density $u_B = B^2 / (8\pi)$. If 100 G unresolved field replenishes every $\tau_{\text{rep}} = 4$ hours, the volumetric power supply per unit area (assuming a scale height $H \sim 150$ km) is
$$F = \frac{u_B \cdot H}{\tau_{\text{rep}}}$$
Compare to the chromospheric radiative loss $\sim 10^7$ erg/cm²/s.

**한국어.** 자기 에너지 밀도 $u_B = B^2 / (8\pi)$. 100 G 미분해 자기장이 $\tau_{\text{rep}} = 4$시간마다 보충되면, 단위 면적당 부피 전력 공급(스케일 높이 $H \sim 150$ km 가정)은
$$F = \frac{u_B \cdot H}{\tau_{\text{rep}}}$$
채층 복사 손실 $\sim 10^7$ erg/cm²/s와 비교.

In [ ]:
"""Estimate magnetic carpet energy budget for a range of unresolved field strengths.

Compares to canonical chromospheric radiative loss.
"""

B_unresolved = np.linspace(20, 200, 100)  # G
TAU_REP_S = 4 * 3600.0  # 4 hours in seconds
H_CM = 1.5e7            # 150 km scale height in cm

u_B = B_unresolved ** 2 / (8 * np.pi)  # erg/cm^3 (cgs Gaussian)
F_carpet = u_B * H_CM / TAU_REP_S  # erg/cm^2/s
F_CHROMO = 1e7  # canonical chromospheric loss

fig, ax = plt.subplots()
ax.plot(B_unresolved, F_carpet, label="Carpet energy flux / 카펫 에너지 플럭스")
ax.axhline(F_CHROMO, color="r", ls="--", label=r"Chromospheric loss $\sim 10^7$ erg/cm$^2$/s")
ax.set_xlabel("Unresolved field $B$ [G] / 미분해 자기장")
ax.set_ylabel(r"Energy flux [erg/cm$^2$/s] / 에너지 플럭스")
ax.set_yscale("log")
ax.set_title("Magnetic carpet energy supply / 자기 카펫 에너지 공급")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

B_threshold = B_unresolved[np.argmin(np.abs(F_carpet - F_CHROMO))]
print(f"Field strength needed to balance chromospheric loss: B ~ {B_threshold:.1f} G")
print(f"At B = 100 G: F_carpet = {(100**2 / (8*np.pi)) * H_CM / TAU_REP_S:.3e} erg/cm^2/s")

## Summary / 요약

**English.** This notebook reproduced three quantitative pillars of the de Wijn et al. (2009) review:
1. The lognormal flux distribution naturally captures both internetwork and ephemeral active region populations; the exponential PDF describes network features specifically.
2. The carpet replacement timescale at Hinode-era emergence rates is hours (resolved field) to days (Hanle-equivalent unresolved field) — orders of magnitude shorter than the global solar cycle, supporting a *local* dynamo origin.
3. The Stokes V cancellation in mixed-polarity pixels demonstrates exactly why Zeeman magnetograms underestimate the true unsigned flux while Hanle depolarisation does not — the resolution of the 1980s–2000s controversy.
4. The unresolved magnetic carpet at $\sim 100$ G can supply enough energy ($\sim 10^6$–$10^7$ erg/cm²/s with $\tau_{\rm rep}=4$ h, $H = 150$ km) to power the quiet chromosphere — making small-scale magnetism a serious candidate for chromospheric heating.

**한국어.** 본 노트북은 de Wijn et al.(2009) 리뷰의 세 가지 정량적 기둥을 재현했습니다:
1. 로그정규 자속 분포는 네트워크간과 단명 활동 영역 개체군을 자연스럽게 포착; 지수 PDF는 네트워크 feature를 특별히 묘사.
2. Hinode 시대 emergence 비율에서 카펫 교체 시간 척도는 시간(분해된 자기장)에서 일(Hanle 환산 미분해 자기장) 단위 — 전역 태양 주기보다 차수 단위 짧아 *국소* 다이나모 기원 지지.
3. 혼합 극성 픽셀에서의 Stokes V 상쇄는 Zeeman magnetogram이 진짜 unsigned 자속을 과소평가하는 반면 Hanle 탈편광은 그렇지 않은 정확한 이유를 시연 — 1980년대–2000년대 논쟁의 해결.
4. $\sim 100$ G의 미분해 자기 카펫은 조용한 채층을 구동할 충분한 에너지($\tau_{\rm rep}=4$ h, $H = 150$ km에서 $\sim 10^6$–$10^7$ erg/cm²/s)를 공급 — 소규모 자기장을 채층 가열의 진지한 후보로 만듭니다.